# K-Nearest Neighbors — From Scratch

A custom KNN classifier implemented as a scikit-learn-compatible estimator (`BaseEstimator`,
`ClassifierMixin`), supporting Euclidean/Manhattan distance and optional distance-weighted
voting. Tuned with `GridSearchCV` and evaluated on the Iris dataset.

In [9]:
!git clone https://github.com/armitakamari/ML-Fundamentals-Implementations.git
%cd ML-Fundamentals-Implementations


Cloning into 'ML-Fundamentals-Implementations'...
remote: Enumerating objects: 30, done.
remote: Counting objects: 100% (30/30), done.
remote: Compressing objects: 100% (28/28), done.
remote: Total 30 (delta 7), reused 0 (delta 0), pack-reused 0 (from 0)
Receiving objects: 100% (30/30), 133.85 KiB | 1.16 MiB/s, done.
Resolving deltas: 100% (7/7), done.
/content/ML-Fundamentals-Implementations/ML-Fundamentals-Implementations/ML-Fundamentals-Implementations


In [10]:
import numpy as np
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report
from sklearn.utils.validation import check_X_y, check_array
import warnings
warnings.filterwarnings("ignore", category=FutureWarning)

## Model

In [11]:
class MyKNNClassifier(BaseEstimator, ClassifierMixin):
    def __init__(self, n_neighbors=3, metric="euclidean", weighted=False):
        self.n_neighbors = n_neighbors
        self.metric = metric
        self.weighted = weighted

    def fit(self, X, y):
        X, y = check_X_y(X, y)
        self.X_train = np.array(X)
        self.y_train = np.array(y)
        return self

    def _compute_distances(self, X):
        X = check_array(X)
        if self.metric == "euclidean":
            dists = np.sqrt(np.sum((X[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]) ** 2, axis=2))
        elif self.metric == "manhattan":
            dists = np.sum(np.abs(X[:, np.newaxis, :] - self.X_train[np.newaxis, :, :]), axis=2)
        else:
            raise ValueError("Unsupported metric: choose \'euclidean\' or \'manhattan\'")
        return dists

    def predict(self, X):
        X = check_array(X)
        dists = self._compute_distances(X)
        predictions = np.zeros(X.shape[0], dtype=int)

        for i in range(X.shape[0]):
            k_idx = np.argsort(dists[i])[:self.n_neighbors]
            k_labels = self.y_train[k_idx]

            if self.weighted:
                weights = 1 / (dists[i][k_idx] + 1e-5)
                label = np.bincount(k_labels, weights=weights).argmax()
            else:
                label = np.bincount(k_labels).argmax()

            predictions[i] = label
        return predictions

## Train / Validation / Test Split, Hyperparameter Search, and Evaluation

In [12]:
X, y = load_iris(return_X_y=True)

X_temp, X_test, y_temp, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)
X_train, X_val, y_train, y_val = train_test_split(
    X_temp, y_temp, test_size=0.25, stratify=y_temp, random_state=42
)

print(f"Train size: {len(X_train)}, Validation size: {len(X_val)}, Test size: {len(X_test)}")

pipeline = Pipeline([
    ("scaler", StandardScaler()),
    ("knn", MyKNNClassifier())
])

param_grid = {
    "knn__n_neighbors": [3, 4, 5, 6, 7, 8, 9],
    "knn__metric": ["euclidean", "manhattan"],
    "knn__weighted": [False, True]
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
grid_search = GridSearchCV(
    pipeline, param_grid, cv=cv, scoring="f1_macro", n_jobs=-1, verbose=1
)

grid_search.fit(X_train, y_train)

print("\n Best Parameters from GridSearchCV")
print(grid_search.best_params_)
print(f"Best cross-validation F1 score: {grid_search.best_score_:.3f}")

best_model = grid_search.best_estimator_
y_pred_test = best_model.predict(X_test)

print("\n")
print("Final Evaluation on TEST DATA:")
print("\n")
print(classification_report(y_test, y_pred_test, digits=3))

Train size: 90, Validation size: 30, Test size: 30
Fitting 5 folds for each of 28 candidates, totalling 140 fits

 Best Parameters from GridSearchCV
{'knn__metric': 'euclidean', 'knn__n_neighbors': 3, 'knn__weighted': True}
Best cross-validation F1 score: 0.967


Final Evaluation on TEST DATA:


              precision    recall  f1-score   support

           0      1.000     1.000     1.000        10
           1      0.909     1.000     0.952        10
           2      1.000     0.900     0.947        10

    accuracy                          0.967        30
   macro avg      0.970     0.967     0.967        30
weighted avg      0.970     0.967     0.967        30

